In [ ]:
"""
Script: Sentinel-2 Annual Composite and Tiled Export
Description:
    This script processes Sentinel-2 Surface Reflectance (SR) imagery for a specified target year.
    It links Cloud Score+ (CS+) for pixel-level cloud and shadow masking, generates seasonal composites
    (prioritizing summer imagery and backfilling gaps with non-summer imagery), creates a pixel source
    tracking layer, and exports the final 10m composite images in parallel tiles to Google Drive.
"""

import ee
import geemap

# Initialize the Earth Engine module.
# Change your own Google Cloud project ID
MY_PROJECT_ID = 'your-google-cloud-project-id'     # your-google-cloud-project-id  
try:
    ee.Initialize(project=MY_PROJECT_ID)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT_ID)
print("GEE Initialized successfully with project ID:", MY_PROJECT_ID) 

In [ ]:
#  Configuration & Parameters   
BANDS_S2 = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

TARGET_YEAR = 2020
START_ID = 111  # tiles ID
END_ID = 111    
EXPORT_FOLDER = 'HMA_S2'

# Study area boundary
area_tb = ee.FeatureCollection('change your own HMA boundary asset path') #change to your own asset path about the study area boundary HMA

In [ ]:
def parse_tile_id(feature):
    """
    des:
        Parses the string 'tile_id' attribute into a numeric property 'tile_id_num'
        to allow numeric range filtering.
    para:
        feature (ee.Feature): An Earth Engine tile feature with a 'tile_id' property.
    return:
        ee.Feature: Feature with the newly assigned 'tile_id_num' numeric property.
    """
    tile_id_num = ee.Number.parse(feature.get('tile_id'))
    return feature.set('tile_id_num', tile_id_num)


def mask_with_cs_plus(image):
    """
    des:
        Masks clouds and cloud shadows using the Cloud Score+ Quality Score ('cs').
    para:
        image (ee.Image): Sentinel-2 image linked with the Cloud Score+ 'cs' band.
    return:
        ee.Image: Cloud-masked Sentinel-2 image retaining clear pixels (cs >= 0.65).
    """
    mask = image.select('cs').gte(0.65)
    return image.updateMask(mask)


def export_tile_to_drive(image, tile_feature, target_year, folder_name):
    """
    des:
        Creates and starts an Earth Engine batch export task to export a 10m GeoTIFF
        for a specific tile to Google Drive.
    para:
        image (ee.Image): The final composite image to be exported.
        tile_feature (dict): Tile GeoJSON dictionary containing geometry and properties.
        target_year (int): Target year used for file naming.
        folder_name (str): Destination folder name in Google Drive.
    return:
        ee.batch.Task: The initiated export task instance.
    """
    tile_props = tile_feature['properties']
    tile_id = str(tile_props.get('tile_id'))
    # Clean whitespace from projection string if present
    proj = str(tile_props.get('proj', '')).replace(" ", "")

    geom = ee.Geometry(tile_feature['geometry'])
    img_tile_name = f'tibet_s2_{target_year}_tile_{tile_id}'

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=img_tile_name,
        folder=folder_name,
        fileNamePrefix=img_tile_name,
        scale=10,
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        region=geom,
        crs=proj
    )
    task.start()
    return task


In [ ]:
#  Load & Filter Spatial Tiles 

tb_tiles_buf = (
    ee.FeatureCollection('change your own HMA tiles asset path') # change to your own asset path about the HMA tiles
    .map(parse_tile_id)
    .filter(
        ee.Filter.And(
            ee.Filter.gte('tile_id_num', START_ID),
            ee.Filter.lte('tile_id_num', END_ID)
        )
    )
    .sort('tile_id_num')
)

#   Load & Cloud-Mask Annual S2 Data 

# Load Sentinel-2 L2A collection
s2_annual = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(area_tb)
    .filter(ee.Filter.calendarRange(TARGET_YEAR, TARGET_YEAR, 'year'))
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
)

# Load Cloud Score+ collection
cs_plus_annual = (
    ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    .filterBounds(area_tb)
    .filter(ee.Filter.calendarRange(TARGET_YEAR, TARGET_YEAR, 'year'))
)

# Link collections and apply cloud masking
s2_with_cs = s2_annual.linkCollection(cs_plus_annual, ['cs'])
s2_masked_annual = s2_with_cs.map(mask_with_cs_plus).select(BANDS_S2)

In [ ]:
#  Split Seasons & Priority Backfill 

# Priority 1: Summer composite (June - September)
s2_summer = (
    s2_masked_annual
    .filter(ee.Filter.calendarRange(6, 9, 'month'))
    .median()
)

# Priority 2: Non-summer composite (January - May OR October - December)
non_summer_filter = ee.Filter.Or(
    ee.Filter.calendarRange(1, 5, 'month'),
    ee.Filter.calendarRange(10, 12, 'month')
)
s2_other_months = (
    s2_masked_annual
    .filter(non_summer_filter)
    .median()
)

# Priority backfill: Fill gaps in the summer composite with other months' data
combined_img = s2_summer.unmask(s2_other_months)
final_img = combined_img.toUint16().unmask(0)

In [ ]:
#  Generate Pixel Source Tracking Layer

source_summer = ee.Image.constant(1).updateMask(s2_summer.select('B2').mask())
source_other = ee.Image.constant(2).updateMask(s2_other_months.select('B2').mask())

# Merge source layers (0 = No data, 1 = Summer, 2 = Other months)
source_map = source_summer.unmask(source_other).unmask(0).toByte()

In [ ]:
#  Interactive Visualization (geemap)
m = geemap.Map()
m.centerObject(area_tb, 5)

vis_true = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
vis_false = {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 3000}
source_vis = {
    'min': 0,
    'max': 2,
    'palette': ['000000', '00FF00', 'FFFF00']
}

# Optional visualization layers (uncomment to display)
# m.addLayer(s2_summer.clipToCollection(area_tb), vis_false, '1. [Ref] Summer (Jun-Sep) False Color', False)
# m.addLayer(final_img.clipToCollection(area_tb), vis_true, '2. [Final] Backfilled True Color', False)
# m.addLayer(final_img.clipToCollection(area_tb), vis_false, '3. [Final] Backfilled False Color', True)
# m.addLayer(source_map.clipToCollection(area_tb), source_vis, '4. Pixel Source (Green=Summer, Yellow=Other)', True)
m.addLayer(tb_tiles_buf, {'color': 'FF0000'}, '5. Tiles Boundary', True)

display(m)

In [ ]:
#  Batch Export to Google Drive

# Fetch tile features locally once to minimize round-trip RPC calls
tiles_data = tb_tiles_buf.getInfo().get('features', [])
num_export = len(tiles_data)

print(f"Total tiles to export: {num_export}")

for tile_fea in tiles_data:
    task = export_tile_to_drive(
        image=final_img,
        tile_feature=tile_fea,
        target_year=TARGET_YEAR,
        folder_name=EXPORT_FOLDER
    )
    tile_id = tile_fea['properties'].get('tile_id')
    print(f"Submitted export task for 10m tile: {tile_id} (Task ID: {task.id})")

print("All export tasks have been submitted. Monitor progress on the Earth Engine Tasks tab or via code.")